In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('Data_Menu.csv')
df.head()

,nama_makanan,kategori,natrium (mg),Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Nasi goreng ayam,makan_pagi,302.0,NaN,NaN,NaN,NaN
1,Nasi goreng,makan_pagi,415.0,NaN,NaN,NaN,NaN
2,Nasi goreng udang,makan_pagi,467.0,NaN,NaN,NaN,NaN
3,Nasi goreng spesial,makan_pagi,703.0,NaN,NaN,NaN,NaN
4,Bubur nasi,makan_pagi,126.0,NaN,NaN,NaN,NaN


In [4]:
# Fungsi kategori hipertensi dan batasan natrium
def get_kategori_natrium(tekanan_darah):
    try:
        sistolik, diastolik = map(int, tekanan_darah.split("/"))
    except:
        raise ValueError("Format tekanan darah harus seperti 140/90")

    if 140 <= sistolik <= 159 or 90 <= diastolik <= 99:
        return "Hipertensi Derajat 1", (1000, 1200)
    elif 160 <= sistolik <= 179 or 100 <= diastolik <= 109:
        return "Hipertensi Derajat 2", (600, 800)
    elif sistolik >= 180 or diastolik >= 110:
        return "Hipertensi Derajat 3", (200, 400)
    else:
        raise ValueError("Tekanan darah tidak termasuk kategori hipertensi.")

In [5]:
# Fungsi rekomendasi utama
def rekomendasi_makanan( nama_makanan, tekanan_darah):
    kategori, (min_natrium, max_natrium) = get_kategori_natrium(tekanan_darah)

    kolom_makanan = ['Makan_Pagi', 'Makan_Siang', 'Makan_Malam', 'Snack_1', 'Snack_2']
    keyword_list = [keyword.lower() for keyword in nama_makanan.split()]

    mask = df[kolom_makanan].apply(
    lambda row: all(keyword in ' '.join(row.dropna().astype(str).str.lower()) for keyword in keyword_list),
    axis=1
    )

    df_kandidat = df[mask]

    if df_kandidat.empty:
        print(f"Tidak ditemukan makanan yang mengandung '{nama_makanan}'.")
        return pd.DataFrame()

    df_filtered = df_kandidat[
        (df_kandidat["Total_Natrium"] >= min_natrium) & (df_kandidat["Total_Natrium"] <= max_natrium)
    ]

    if df_filtered.empty:
        print(f"Tidak ditemukan makanan '{nama_makanan}' dalam rentang natrium {min_natrium}–{max_natrium}.")
        return pd.DataFrame()

    natrium_input = df_filtered.iloc[0]["Total_Natrium"]

    # Standarisasi
    scaler = StandardScaler()
    df_filtered["Total_Natrium_Scaled"] = scaler.fit_transform(df_filtered[["Total_Natrium"]])

    X_filtered = df_filtered[['Total_Natrium_Scaled']].values
    k = min(10, len(X_filtered))  # Dinamis sesuai jumlah data
    if k < 1:
        print("Jumlah data terlalu sedikit untuk rekomendasi.")
        return pd.DataFrame()

    knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    knn.fit(X_filtered)

    natrium_input_scaled = scaler.transform([[natrium_input]])
    distances, indices = knn.kneighbors(natrium_input_scaled)

    hasil = []
    for idx, distance in zip(indices[0], distances[0]):
        row = df_filtered.iloc[idx]
        hasil.append({
            "Kategori Hipertensi": kategori,
            "Makan Pagi": row["Makan_Pagi"],
            "Makan Siang": row["Makan_Siang"],
            "Makan Malam": row["Makan_Malam"],
            "Snack 1": row["Snack_1"],
            "Snack 2": row["Snack_2"],
            "Total Natrium": row["Total_Natrium"]
        })

    return pd.DataFrame(hasil)


In [6]:
# Misalkan kita ingin mencari makanan berdasarkan nama 'tempe' dan tekanan darah 120
nama_makanan_input = "ikan"
tekanan_darah_input = "180/110"

# Memanggil fungsi rekomendasi_makanan dengan input nama makanan dan tekanan darah
df_rekomendasi = rekomendasi_makanan(nama_makanan_input, tekanan_darah_input)

# Menampilkan hasil rekomendasi makanan
if not df_rekomendasi.empty:
    print("Rekomendasi Makanan:")
    print(df_rekomendasi)
else:
    print("Tidak ada rekomendasi makanan yang ditemukan.")



KeyError: "None of [Index(['Makan_Pagi', 'Makan_Siang', 'Makan_Malam', 'Snack_1', 'Snack_2'], dtype='object')] are in the [columns]"

In [7]:
# Panggil fungsi rekomendasi
hasil = rekomendasi_makanan(nama_makanan, tekanan_darah)
print(hasil)

NameError: name 'nama_makanan' is not defined